In [1]:
import sys
from pathlib import Path
 
print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)
 
for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.13.5
Folder: Downloads
numpy - ok
pandas - ok
sklearn - ok


In [2]:
import csv
from pathlib import Path
import numpy as np
 
SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)
 
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path
 
 
if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)


dataset ready: data\delivery_times.csv


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
 
orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X = orders[FEATURES]
y = orders["delivery_min"]
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

480 120


In [4]:
from sklearn.metrics import mean_absolute_error
 
def score_both_ways(model, name):
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    gap = test_mae - train_mae
    print(name, "train", round(train_mae, 2), "test", round(test_mae, 2), "gap", round(gap, 2))
    return {"name": name, "train": train_mae, "test": test_mae, "gap": gap}

In [5]:
from sklearn.linear_model import LinearRegression
 
linear = score_both_ways(LinearRegression(), "LinearRegression")

LinearRegression train 2.04 test 1.92 gap -0.11


In [6]:
from sklearn.tree import DecisionTreeRegressor
 
wild_tree = score_both_ways(DecisionTreeRegressor(random_state=42), "DecisionTree (no limit)")

DecisionTree (no limit) train 0.0 test 3.43 gap 3.43


In [7]:
small_tree = score_both_ways(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")

DecisionTree (depth 4) train 3.66 test 4.23 gap 0.57


In [8]:
from sklearn.ensemble import RandomForestRegressor
 
forest = score_both_ways(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

RandomForest (50 trees) train 1.0 test 2.39 gap 1.4


In [9]:
results = pd.DataFrame([linear, wild_tree, small_tree, forest]).round(2).sort_values("test")
print(results.to_string(index=False))


                   name  train  test   gap
       LinearRegression   2.04  1.92 -0.11
RandomForest (50 trees)   1.00  2.39  1.40
DecisionTree (no limit)   0.00  3.43  3.43
 DecisionTree (depth 4)   3.66  4.23  0.57


In [10]:
from sklearn.model_selection import cross_val_score
 
def cross_validate(model, name):
    scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(name, "MAE", round(scores.mean(), 2))
    return scores.mean()
 
cv_linear = cross_validate(LinearRegression(), "LinearRegression")
cv_tree = cross_validate(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")
cv_forest = cross_validate(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

LinearRegression MAE 2.03
DecisionTree (depth 4) MAE 4.57
RandomForest (50 trees) MAE 2.69


In [11]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}
 
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X, y, cv=5,
                             scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    scores_by_depth[depth] = round(float(mae), 3)
 
best_depth = min(scores_by_depth, key=scores_by_depth.get)
print(scores_by_depth)
print("best depth:", best_depth)

{2: 5.858, 3: 4.85, 4: 4.573, 6: 3.642, 8: 3.395, None: 3.478}
best depth: 8


In [12]:
ranking = sorted(
    {"LinearRegression": cv_linear, "DecisionTree(4)": cv_tree, "RandomForest(50)": cv_forest}.items(),
    key=lambda kv: kv[1],
)
for name, mae in ranking:
    print(name, round(mae, 2))

LinearRegression 2.03
RandomForest(50) 2.69
DecisionTree(4) 4.57


In [13]:
import os
# To Do: Breast Cancer Classification

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

# ---------------------------------------------------------
# 1. Load the Breast Cancer dataset
# ---------------------------------------------------------
# The Kaggle dataset is based on the Wisconsin Breast Cancer dataset.
# If a Kaggle CSV is available locally, place it at data/breast_cancer.csv.
# Otherwise, sklearn's copy of the same Wisconsin dataset is used.

possible_files = [
    "data/breast_cancer.csv",
    "breast_cancer.csv",
    "data/data.csv",
    "data.csv"
]

csv_path = next((p for p in possible_files if os.path.exists(p)), None)

if csv_path:
    cancer = pd.read_csv(csv_path)
    # Kaggle's common dataset uses 'diagnosis' as target and may contain an ID column.
    target_col = "diagnosis" if "diagnosis" in cancer.columns else cancer.columns[-1]
    cancer = cancer.drop(columns=["id", "ID"], errors="ignore")
    X = cancer.drop(columns=[target_col])
    y = cancer[target_col]

    # Convert categorical diagnosis (M/B) to 1/0 if necessary.
    if y.dtype == "object":
        y = y.astype(str).str.strip().map({"M": 1, "B": 0}).fillna(
            pd.Categorical(y).codes
        )
    y = pd.Series(y).astype(int)
else:
    data = load_breast_cancer(as_frame=True)
    X = data.data
    y = data.target

print("Dataset shape:", X.shape)
print("Classes:", sorted(y.unique()))

# ---------------------------------------------------------
# 2. Train/test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

# ---------------------------------------------------------
# 3. Define classifiers
# ---------------------------------------------------------
# Scaling is important for Logistic Regression.
logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

tree = DecisionTreeClassifier(random_state=42)
forest = RandomForestClassifier(
    n_estimators=100, random_state=42
)

models = {
    "Logistic Regression": logistic,
    "Decision Tree": tree,
    "Random Forest": forest
}

# ---------------------------------------------------------
# 4. Train models and compare train/test accuracy
# ---------------------------------------------------------
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    results.append({
        "Model": name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Gap": train_acc - test_acc
    })

accuracy_results = pd.DataFrame(results).sort_values(
    "Test Accuracy", ascending=False
)

print("\nTrain/Test Accuracy:")
print(accuracy_results.round(4).to_string(index=False))

# ---------------------------------------------------------
# 5. Cross-validation to find the best Decision Tree depth
# ---------------------------------------------------------
depths = [2, 3, 4, 5, 6, 8, 10, None]
depth_scores = {}

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=5, scoring="accuracy")
    depth_scores[depth] = scores.mean()

best_depth = max(depth_scores, key=depth_scores.get)

print("\nDecision Tree depth CV accuracy:")
for depth, score in depth_scores.items():
    print(f"depth={depth}: {score:.4f}")

print("Best tree depth:", best_depth)
print("Best CV accuracy:", round(depth_scores[best_depth], 4))

# Train the best-depth tree and compare it with the other models.
best_tree = DecisionTreeClassifier(
    max_depth=best_depth, random_state=42
)
best_tree.fit(X_train, y_train)

best_tree_test_pred = best_tree.predict(X_test)
best_tree_acc = accuracy_score(y_test, best_tree_test_pred)

print("Best-depth Decision Tree test accuracy:",
      round(best_tree_acc, 4))

# ---------------------------------------------------------
# 6. Confusion matrix and classification metrics
# ---------------------------------------------------------
# Use the best-depth tree in place of the original tree if it improves it.
final_models = {
    "Logistic Regression": logistic,
    "Decision Tree (best depth)": best_tree,
    "Random Forest": forest
}

metric_rows = []

print("\nConfusion Matrices and Classification Reports:")

for name, model in final_models.items():
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)

    print(f"\n{name}")
    print("Confusion Matrix:")
    print(cm)
    print(classification_report(y_test, pred, digits=4))

    metric_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0)
    })

metrics = pd.DataFrame(metric_rows).sort_values(
    "F1 Score", ascending=False
)

print("\nFinal Classification Metrics:")
print(metrics.round(4).to_string(index=False))

best_model_name = metrics.iloc[0]["Model"]
print("\nBest-performing model:", best_model_name)


Dataset shape: (569, 30)
Classes: [np.int64(0), np.int64(1)]
Training samples: 455
Testing samples : 114

Train/Test Accuracy:
              Model  Train Accuracy  Test Accuracy    Gap
Logistic Regression           0.989         0.9825 0.0066
      Random Forest           1.000         0.9561 0.0439
      Decision Tree           1.000         0.9123 0.0877

Decision Tree depth CV accuracy:
depth=2: 0.9187
depth=3: 0.9253
depth=4: 0.9385
depth=5: 0.9319
depth=6: 0.9187
depth=8: 0.9099
depth=10: 0.9099
depth=None: 0.9099
Best tree depth: 4
Best CV accuracy: 0.9385
Best-depth Decision Tree test accuracy: 0.9386

Confusion Matrices and Classification Reports:

Logistic Regression
Confusion Matrix:
[[41  1]
 [ 1 71]]
              precision    recall  f1-score   support

           0     0.9762    0.9762    0.9762        42
           1     0.9861    0.9861    0.9861        72

    accuracy                         0.9825       114
   macro avg     0.9812    0.9812    0.9812       114
weight